# Canonical Model 06: Ensemble Calibration & Uncertainty (PESTPP-IES)

This notebook runs **iterative ensemble smoother** history matching with one line — `cal.run_ies(...)` — and then assesses the outcome with `IesResults`: phi convergence, prior/posterior ensembles against the observations, and **posterior forecast uncertainty** ('history matching for nothing, uncertainty analysis for free').

It reuses the compact calibration demo and the same declarative setup as Notebook 05.

In [1]:
import sys
from pathlib import Path

# Make the in-repo `src/` importable when myflopy is not pip-installed.
src = Path.cwd().parents[2] / "src"
if src.exists() and str(src) not in sys.path:
    sys.path.insert(0, str(src))

import pandas as pd
import myflopy as mf
from canonical_notebook_style import notebook_header
from modern_pest_demo import build_calibration_demo
from myflopy.modflow.mf6.pest import PestProject

notebook_header(
    '06',
    'Ensemble Calibration & Uncertainty',
    'Run PESTPP-IES and assess fit, ensembles, and posterior forecast uncertainty.',
)

## 1. Build and parameterize the demo (same as Notebook 05)

A constant K multiplier and a per-cell recharge field, head observations, and one forecast.

In [2]:
artifact_root = Path('../artifacts/canonical_pest_modern')
artifact_root.mkdir(parents=True, exist_ok=True)

demo = build_calibration_demo(artifact_root / 'ies_model')

cal = PestProject(
    model=demo.model,
    name='ies_demo',
    workspace=artifact_root / 'ies_template',
    start_datetime='2024-01-01',
)
cal.parameterize('k',        style='constant', bounds=(0.05, 2.0), physical=(0.01, 100.0),
                 capture=True)  # record the resolved K field for spatial maps
cal.parameterize('recharge', style='grid',     bounds=(0.5, 1.5),  physical=(0.0, 1e-2))
cal.observe(demo.head_targets)
cal.forecast(demo.forecast_targets)
cal.build('ies_demo.pst', noptmax=0)
print(cal.settings())

VoronoiGrid initializing.
Voronoi grid initialized.
writing simulation...
  writing simulation name file...
  writing simulation tdis package...
  writing solution package ims_gwf...
  writing model pest_demo...
    writing model name file...
    writing package disv...
    writing package ic...
    writing package npf...
    writing package sto...
    writing package oc...
    writing package chd...
INFORMATION: maxbound in ('', 'chd', 'dimensions') changed to 24 based on size of stress_period_data
    writing package rch...

Saved model object to .model file: ..\artifacts\canonical_pest_modern\ies_model\pest_demo\pest_demo.model

FloPy is using the following executable to run the model: ..\..\..\..\..\..\..\..\PATH\modflow_exe\mf6.exe
                                   MODFLOW 6
                U.S. GEOLOGICAL SURVEY MODULAR HYDROLOGIC MODEL
                            VERSION 6.7.0 02/05/2026

   MODFLOW 6 compiled Feb 05 2026 22:36:44 with Intel(R) Fortran Intel(R) 64
   Compiler C

    Solving:  Stress period:     1    Time step:     1
 
 Run end date and time (yyyy/mm/dd hh:mm:ss): 2026/06/22  9:57:57
 Elapsed run time:  0.206 Seconds
 
 Normal termination of simulation.

Success is:  True
writing simulation...
  writing simulation name file...
  writing simulation tdis package...
  writing solution package ims_gwf...
  writing model pest_demo...
    writing model name file...
    writing package disv...
    writing package ic...
    writing package npf...
    writing package sto...
    writing package oc...
    writing package chd...
    writing package rch...

Error saving model object to .model file: cannot pickle 'BufferedReader' instances

FloPy is using the following executable to run the model: ..\..\..\..\..\..\..\..\PATH\modflow_exe\mf6.exe
                                   MODFLOW 6
                U.S. GEOLOGICAL SURVEY MODULAR HYDROLOGIC MODEL
                            VERSION 6.7.0 02/05/2026

   MODFLOW 6 compiled Feb 05 2026 22:36:44 with Intel

    Solving:  Stress period:     1    Time step:     1
 
 Run end date and time (yyyy/mm/dd hh:mm:ss): 2026/06/22  9:57:57
 Elapsed run time:  0.080 Seconds
 
 Normal termination of simulation.

Success is:  True


C:\Users\lukem\Python\mf-env\.venv\Lib\site-packages\pyemu\utils\pst_from.py:1281: PyemuWarning: add_py_function(): _write_head_target_csv already in forward run python functions, not overriding here, original will be maintained


PEST calibration: ies_demo  (model: pest_demo)
  template : ..\artifacts\canonical_pest_modern\ies_template
  start    : 2024-01-01
  parameters (2):
    - k          style=constant    bounds=(0.05, 2.0) physical=(0.01, 100.0) transform=log
    - recharge   style=grid        bounds=(0.5, 1.5) physical=(0.0, 0.01) transform=log
  observations (1):
    - hds        kind=headtarget   n=8
  forecasts (1):
    - fore1      n=1
  built control file:
    npar=145 (groups=2)  nobs=153 (nonzero weight=8)  forecasts=1  noptmax=0


## 2. Run PESTPP-IES

`run_ies` configures the ensemble options, launches PESTPP-IES, and returns an `IesResults`. Defaults make `cal.run_ies()` a valid first call; we use a small ensemble here so the notebook runs quickly. For a real study, raise `reals` (100–300) and pass `workers=N` for parallel agents.

> The PEST++ executable is found next to your MODFLOW 6 binary; no manual PATH setup is needed.

In [3]:
RUN_IES = True   # set False to skip the (slow) ensemble run
REALS = 20
ITERATIONS = 2

if RUN_IES:
    ies = cal.run_ies(reals=REALS, iterations=ITERATIONS)
    print(ies.settings)
else:
    ies = None
    print('Skipped: set RUN_IES = True to run PESTPP-IES.')

pestpp-ies.exe ies_demo.pst


PESTPP-IES run: ies_demo
  workspace   : ..\artifacts\canonical_pest_modern\ies_template
  realizations: 20
  noptmax     : 2   iterations on disk: [0, 1, 2]
  observations: 8 nonzero-weight   forecasts: 1
  noise ensemble: yes
  ies options : {'ies_num_reals': 20}


## 3. Phi convergence

The first thing to check: did history matching reduce the misfit? Each faint line is one realization; the bold line is the ensemble mean.

In [4]:
if ies is not None:
    display(ies.plot_phi())

## 4. Ensemble versus observations

Grey is the prior ensemble, blue the posterior, red the measured values. A good fit means the blue spread brackets the red markers.

In [5]:
if ies is not None:
    display(ies.plot_vs_obs())

## 5. Forecast uncertainty

The payoff: the posterior distribution of each prediction of interest. `forecasts()` summarizes prior→posterior uncertainty for every forecast; `forecast(name).plot()` shows the prior (grey) and posterior (blue) histograms with the known value marked.

In [6]:
if ies is not None:
    display(ies.forecasts())
    name = ies.forecast_names[0]
    display(ies.forecast(name).plot())

,prior_mean,prior_std,prior_p05,prior_p50,prior_p95,posterior_mean,posterior_std,posterior_p05,posterior_p50,posterior_p95,truth,uncertainty_reduction
forecast,,,,,,,,,,,,
oname:fore1_otype:lst_usecol:pred_00_per:0,32.030072,0.207868,31.838069,31.947849,32.469643,32.239923,0.22955,31.981326,32.172285,32.663032,32.812385,-0.104306


## 6. The single best realization

If you must carry one parameter set forward, use the **base** (minimum-error-variance) realization — `best()` returns it. Resist the temptation to pick the lowest-phi realization; it tends to be over-fit.

In [7]:
if ies is not None:
    print('recommended realization:', ies.best())
    print('iterations on disk:', ies.iterations)

recommended realization: base
iterations on disk: [0, 1, 2]


## 6b. Spatial parameter maps ("property patterns")

Because we declared `capture=True`, every realization's resolved K field is recorded, so we can map it on the grid. This answers the workshop's key question — *"property patterns: plausible or laughable?"* — and shows where history matching actually moved the conductivity.

- **mean**: the posterior-mean K field
- **std**: where K is still uncertain after history matching
- **change**: posterior-mean / prior-mean — where (and how much) calibration moved K

In [8]:
if ies is not None:
    display(ies.plot_field('k', stat='mean', which='posterior'))
    display(ies.plot_field('k', stat='std'))
    display(ies.plot_field('k', stat='change'))
    display(ies.field('k').head())

,cell,prior_mean,prior_std,posterior_mean,posterior_std,base,change
0,0,10.6261,5.832264,5.350335,2.295356,5.04796,0.503509
1,1,10.6261,5.832264,5.350335,2.295356,5.04796,0.503509
2,2,10.6261,5.832264,5.350335,2.295356,5.04796,0.503509
3,3,10.6261,5.832264,5.350335,2.295356,5.04796,0.503509
4,4,10.6261,5.832264,5.350335,2.295356,5.04796,0.503509


## 7. One-shot HTML report

`report(...)` bundles phi convergence, the ensemble-vs-observation comparison, and a histogram per forecast into a single self-contained HTML file — the ensemble analogue of the deterministic review export.

In [9]:
if ies is not None:
    report_path = ies.report(artifact_root / 'ies_review.html')
    print('wrote', report_path)

wrote ..\artifacts\canonical_pest_modern\ies_review.html


## Interpretation checklist

- Confirm phi actually dropped, and that the ensemble did not collapse to a single line (which signals over-fitting / spurious correlation).
- Check the posterior brackets the measured data without being implausibly narrow.
- For forecasts, a *good fit is not the same as a good prediction* — value the posterior spread, not just a reduced phi. More realizations give more robust uncertainty estimates.
- Carry the **base** realization forward, never the lowest-phi one.